# Tutorial 3 - Question 6
## Nonlinear regression for the toluene hydrogenation rate law

Reaction: `T + H2 -> B + M`. The goal is to use the differential-reactor data to test a rate law and estimate the kinetic parameters.

**Important notation:** in the table, `pH` means the partial pressure of hydrogen, H2. It is not hydrogen pressure squared. The data themselves prove this: when pH2 changes from 1 to 2 to 4 atm, the rate changes from about 71.8 to 142 to 284, which is first order in H2.

## 1. Model and physical meaning

We will fit

`rate = k * pT * pH / (1 + KB * pB + KT * pT)`

Here `k` is a lumped kinetic constant, `KB` measures how strongly benzene blocks catalyst sites, and `KT` measures how strongly toluene occupies sites. Methane is not included in the denominator because changing methane from 0 to 4 atm barely changes the measured rate.

The denominator is a site balance: vacant sites plus T-occupied sites plus B-occupied sites.

In [ ]:
# NumPy stores the experimental columns as arrays and makes calculations convenient.
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# The table reports rate numbers multiplied by 10^10.
# We first fit those displayed numbers; this avoids very small floating-point values.
rate_table = np.array([71, 71.3, 41.6, 19.7, 42, 17.1, 71.8, 142, 284, 47, 71.3, 117, 127, 131, 133, 41.8], dtype=float)

# Partial pressures, in atm, copied row-by-row from the tutorial table.
pT = np.array([1, 1, 1, 1, 1, 1, 1, 1, 1, 0.5, 1, 5, 10, 15, 20, 1], dtype=float)
pH = np.array([1, 1, 1, 1, 1, 1, 1, 2, 4, 1, 1, 1, 1, 1, 1, 1], dtype=float)
pM = np.array([1, 4, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], dtype=float)
pB = np.array([0, 0, 1, 4, 1, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], dtype=float)

print('Number of experiments:', len(rate_table))
print('First three observed rates:', rate_table[:3])

## 2. Inspect the experimental trends

Before fitting, we look for the order in each reactant and for inhibition. This prevents us from asking a numerical optimizer to discover a basic pattern that we can identify chemically.

In [ ]:
# Runs 7, 8, and 9 vary only hydrogen pressure.
print('Hydrogen-pressure runs:')
for i in [6, 7, 8]:
    print(f'run {i+1}: pH2={pH[i]:g} atm, rate={rate_table[i]:g}')

# Runs 3, 4, and 6 vary benzene pressure at pT=pH=1 atm.
print('\nBenzene-pressure runs:')
for i in [2, 3, 5]:
    print(f'run {i+1}: pB={pB[i]:g} atm, rate={rate_table[i]:g}')

# Runs 12-15 vary toluene pressure.
print('\nToluene-pressure runs:')
for i in [11, 12, 13, 14]:
    print(f'run {i+1}: pT={pT[i]:g} atm, rate={rate_table[i]:g}')

## 3. Define the nonlinear model

`curve_fit` will choose `k`, `KB`, and `KT` so that the predicted rates are close to the observations. The function receives all three pressures together and returns one predicted rate for each experiment.

In [ ]:
def rate_model(pressures, k, KB, KT):
    # Unpack the pressure arrays in the same order used by the call below.
    toluene, hydrogen, benzene = pressures

    # This is the proposed Langmuir-Hinshelwood form.
    # pH is p(H2), so it appears to the first power.
    denominator = 1 + KB * benzene + KT * toluene
    return k * toluene * hydrogen / denominator

# Positive bounds are physically sensible for this irreversible-rate model.
initial_guess = [150, 1.4, 1.0]
parameters, covariance = curve_fit(
    rate_model,
    (pT, pH, pB),
    rate_table,
    p0=initial_guess,
    bounds=(0, np.inf),
    maxfev=100000,
)

k, KB, KT = parameters
predicted = rate_model((pT, pH, pB), k, KB, KT)
print(f'k  = {k:.6f} in table-rate units')
print(f'KB = {KB:.6f} 1/atm')
print(f'KT = {KT:.6f} 1/atm')

## 4. Measure how well the model fits

RMSE (root mean square error) is the typical prediction error in the table's rate units. R² compares the model with a baseline that always predicts the mean rate; values close to 1 indicate an excellent fit.

In [ ]:
residuals = rate_table - predicted
rmse = np.sqrt(np.mean(residuals**2))
r_squared = 1 - np.sum(residuals**2) / np.sum((rate_table - rate_table.mean())**2)
print(f'RMSE = {rmse:.6f} table-rate units')
print(f'R^2  = {r_squared:.8f}')

# A row-by-row comparison helps us catch data-entry errors.
comparison = np.column_stack((np.arange(1, 17), rate_table, predicted, residuals))
print('\n run   observed   predicted   residual')
for row in comparison:
    print(f'{int(row[0]):3d} {row[1]:10.3f} {row[2]:10.3f} {row[3]:10.3f}')

## 5. Plot observed versus predicted rates

If the model is appropriate, points should lie close to the 45-degree line. A systematic curve away from that line would suggest a missing pressure effect or an unsuitable mechanism.

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(rate_table, predicted, color='teal', edgecolor='black', label='16 experiments')
limit = max(rate_table.max(), predicted.max()) * 1.05
plt.plot([0, limit], [0, limit], '--', color='gray', label='perfect prediction')
plt.xlabel('Observed rate (table units)')
plt.ylabel('Predicted rate (table units)')
plt.title('Nonlinear regression quality check')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. Convert the fitted constant to the unscaled rate unit

The table uses `rate_table = actual_rate * 10^10`. Therefore the physical rate is `rate_table * 10^-10`, and the fitted k must also be multiplied by `10^-10`.

Final result:

- `k = 144.7673 × 10^-10 = 1.4477 × 10^-8 gmol/(g-cat s atm²)`
- `KB = 1.3905 atm^-1`
- `KT = 1.0384 atm^-1`

Rounded values: `k ≈ 145 × 10^-10`, `KB ≈ 1.39 atm^-1`, `KT ≈ 1.04 atm^-1`. These agree with the compact rate law reported in the tutorial.

## 7. Try it yourself

Change the model to use `hydrogen**2` instead of `hydrogen`, fit again, and compare R². The fit becomes poor because the data show first-order dependence on H2. Then restore the correct model.